# Lab 22 — Final Research Capstone

*AI Cybersecurity Masterclass · Lab 22 · AI60secLab*
## Adaptive Cyber Threat Detection Under Shift and Adversarial Conditions

**Core research question:** How does a machine-learning cyber threat detector behave when the training data, input distribution, or decision environment changes, and which defensive intervention best preserves useful detection performance?

The capstone combines:
1. clean baseline;
2. validation-selected threshold;
3. distribution-shift measurement;
4. bounded input stress;
5. training-data poisoning sensitivity;
6. robustness mitigation.

In [ ]:
# Setup: works in Google Colab and locally. No uploads needed.
# In Colab this clones the course repo, which already includes the UNSW-NB15 data.
from pathlib import Path
import sys, os

REPO_URL = "https://github.com/Jacquelinepersha/ai-cybersecurity-public-labs.git"
REPO_NAME = "ai-cybersecurity-public-labs"

try:
    import google.colab  # noqa: F401
    if not Path(REPO_NAME).exists():
        !git clone --depth 1 {REPO_URL}
    os.chdir(REPO_NAME)
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data folder :", DATA_DIR)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

from src.data_loader import load_unsw
from src.preprocessing import split_xy, align_columns
from src.features import drop_identifier_like_columns
from src.models import logistic_pipeline
from src.evaluation import binary_metrics
from src.thresholds import (
    threshold_sweep,
    choose_threshold,
    apply_threshold,
)
from src.drift import domain_classifier_auc
from src.adversarial import (
    available_numeric_attack_features,
    training_bounds,
    bounded_random_perturbation,
    augment_attacks_with_noise,
)
from src.poisoning import random_label_flip

train, test = load_unsw(DATA_DIR)

X_full, y_full = split_xy(train, "label")
X_test, y_test = split_xy(test, "label")
X_full, X_test = align_columns(X_full, X_test)

X_full = drop_identifier_like_columns(X_full)
X_test = drop_identifier_like_columns(X_test)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_full,
    y_full,
    test_size=0.20,
    stratify=y_full,
    random_state=42,
)

In [ ]:
baseline = logistic_pipeline(X_train)
baseline.fit(X_train, y_train)

validation_scores = baseline.predict_proba(X_valid)[:, 1]
sweep = threshold_sweep(
    y_valid,
    validation_scores,
    beta=2.0,
)
selected_threshold, _ = choose_threshold(
    sweep,
    objective="f_beta",
)

test_scores = baseline.predict_proba(X_test)[:, 1]

default_metrics = binary_metrics(
    y_test,
    apply_threshold(test_scores, 0.50),
    test_scores,
)
tuned_metrics = binary_metrics(
    y_test,
    apply_threshold(test_scores, selected_threshold),
    test_scores,
)

print("Selected threshold:", round(selected_threshold, 3))

In [ ]:
domain_auc, _ = domain_classifier_auc(
    X_train,
    X_test,
    sample_per_domain=15000,
    random_state=42,
)

print(
    "Train-vs-test domain classifier AUC:",
    round(domain_auc, 4),
)

In [ ]:
stress_columns = available_numeric_attack_features(X_train)
bounds = training_bounds(X_train, stress_columns)

X_stressed = bounded_random_perturbation(
    X_test,
    columns=stress_columns,
    epsilon=0.10,
    bounds=bounds,
    random_state=99,
)

stressed_scores = baseline.predict_proba(X_stressed)[:, 1]
stressed_metrics = binary_metrics(
    y_test,
    apply_threshold(
        stressed_scores,
        selected_threshold,
    ),
    stressed_scores,
)

In [ ]:
poisoned_y = random_label_flip(
    y_train,
    fraction=0.05,
    random_state=42,
)

poisoned_model = logistic_pipeline(X_train)
poisoned_model.fit(X_train, poisoned_y)

poisoned_scores = poisoned_model.predict_proba(X_test)[:, 1]
poisoned_metrics = binary_metrics(
    y_test,
    apply_threshold(
        poisoned_scores,
        selected_threshold,
    ),
    poisoned_scores,
)

In [ ]:
X_augmented, y_augmented = augment_attacks_with_noise(
    X_train,
    y_train,
    columns=stress_columns,
    epsilon=0.05,
    copies=1,
    bounds=bounds,
    random_state=42,
)

robust_model = logistic_pipeline(X_augmented)
robust_model.fit(X_augmented, y_augmented)

robust_clean_scores = robust_model.predict_proba(X_test)[:, 1]
robust_stress_scores = robust_model.predict_proba(X_stressed)[:, 1]

robust_clean_metrics = binary_metrics(
    y_test,
    apply_threshold(
        robust_clean_scores,
        selected_threshold,
    ),
    robust_clean_scores,
)

robust_stress_metrics = binary_metrics(
    y_test,
    apply_threshold(
        robust_stress_scores,
        selected_threshold,
    ),
    robust_stress_scores,
)

In [ ]:
def metric_row(experiment, metrics):
    return {
        "experiment": experiment,
        "precision": metrics.precision,
        "recall": metrics.recall,
        "f1": metrics.f1,
        "false_positive_rate": metrics.false_positive_rate,
        "false_negative_rate": metrics.false_negative_rate,
        "roc_auc": metrics.roc_auc,
    }

final_results = pd.DataFrame([
    metric_row(
        "Baseline default threshold",
        default_metrics,
    ),
    metric_row(
        "Baseline validation-tuned threshold",
        tuned_metrics,
    ),
    metric_row(
        "Tuned model under input stress",
        stressed_metrics,
    ),
    metric_row(
        "5% label-poisoned training",
        poisoned_metrics,
    ),
    metric_row(
        "Augmented model on clean test",
        robust_clean_metrics,
    ),
    metric_row(
        "Augmented model under stress",
        robust_stress_metrics,
    ),
])

display(final_results.round(4))
print(
    "Independent distribution-shift signal — domain classifier AUC:",
    round(domain_auc, 4),
)

# Final white-paper structure

1. **Abstract**
2. **Research question and falsifiable hypotheses**
3. **Dataset and limitations**
4. **Methods and train/validation/test separation**
5. **Clean results**
6. **Shift, stress, and poisoning results**
7. **Mitigation comparison**
8. **Security interpretation**
9. **Limitations**
10. **Reproducibility and future research**

**Final principle:** do not claim “the model is secure.” State exactly what conditions were tested, what degraded, what improved, and what remains unknown.